In [ ]:
import numpy as np
import pickle
from matplotlib import pyplot as plt

In [ ]:
with open("results/results_run_028749/window_0/window_000_3_303s_lrt_plaw_vs_exp_plaw_boot_df.pkl", "rb") as f:
    boot_df_lrt_028749_0 = pickle.load(f)
print(boot_df_lrt_028749_0.keys())
with open("results/results_run_028749/window_000_3_303s_lrt_plaw_vs_powerlaw_summary.pkl", "rb") as f:
    pp_summary = pickle.load(f)
with open("results/results_run_028749/window_000_3_303s_lrt_plaw_vs_powerlaw_boot_df.pkl", "rb") as f:
    pp_boot_df = pickle.load(f)
with open("results/results_run_028749/window_000_3_303s_lrt_plaw_vs_expadd_summary.pkl", "rb") as f:
    padd_summary = pickle.load(f)
with open("results/results_run_028749/window_000_3_303s_lrt_plaw_vs_expadd_boot_df.pkl", "rb") as f:
    padd_boot_df = pickle.load(f)
with open("results/results_run_028749/window_000_3_303s_lrt_plaw_vs_multi_exp_summary.pkl", "rb") as f:
    pmexp_summary = pickle.load(f)
with open("results/results_run_028749/window_000_3_303s_lrt_plaw_vs_multi_exp_boot_df.pkl", "rb") as f:
    pmexp_boot_df = pickle.load(f)
with open("results/results_run_028749/window_000_3_303s_lrt_plaw_vs_exp_plaw_summary.pkl", "rb") as f:
    pexpp_summary = pickle.load(f)
with open("results/results_run_028749/window_000_3_303s_lrt_plaw_vs_exp_plaw_boot_df.pkl", "rb") as f:
    pexpp_boot_df = pickle.load(f)
with open("results/results_run_028749/window_000_3_303s_lrt_plaw_vs_pure_exp_summary.pkl", "rb") as f:
    ppexp_summary = pickle.load(f)
with open("results/results_run_028749/window_000_3_303s_lrt_plaw_vs_pure_exp_boot_df.pkl", "rb") as f:
    ppexp_boot_df = pickle.load(f)

In [ ]:
t_switch = []
tau = []
for i in boot_df_lrt_028749_0["values_exp_plaw"]:
    t_switch.append(i[3])
    tau.append(i[2])
t_switch_mean = np.mean(t_switch)
tau_mean = np.mean(tau)

In [ ]:
TS_pp = pp_boot_df[pp_boot_df["success"]]["TS"]
TS_padd = padd_boot_df[padd_boot_df["success"]]["TS"]
TS_pmexp = pmexp_boot_df[pmexp_boot_df["success"]]["TS"]
TS_ppexp = ppexp_boot_df[ppexp_boot_df["success"]]["TS"]
TS_pexpp = pexpp_boot_df[pexpp_boot_df["success"]]["TS"]

In [ ]:
def plot_lrt(
    boot_df_or_TS,
    summary,
    null_label="Power law",
    alt_label="Alternative",
    bins=30,
    ax=None,
):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))
    else:
        fig = ax.figure

    if hasattr(boot_df_or_TS, "columns"):
        TS_boot = boot_df_or_TS.loc[boot_df_or_TS["success"], "TS"].to_numpy(dtype=float)
    else:
        TS_boot = np.asarray(boot_df_or_TS, dtype=float)

    TS_boot = TS_boot[np.isfinite(TS_boot)]
    TS_real = float(summary["TS_real"])

    n_boot = int(summary.get("N_boot", len(TS_boot)))
    n_exceed = int(summary.get("N_exceed", np.sum(TS_boot >= TS_real)))
    p_plus1 = float(summary.get("p_plus1", (n_exceed + 1) / (len(TS_boot) + 1)))

    ax.hist(
        TS_boot,
        bins=bins,
        histtype="stepfilled",
        alpha=0.45,
        edgecolor="black",
        label="TS under null",
    )

    ax.axvline(
        TS_real,
        linewidth=2,
        label=fr"Observed TS = {TS_real:.3g}",
    )

    ax.set_title(f"LRT : {null_label} vs. {alt_label}")
    ax.set_xlabel(r"Test statistic, $\Delta(-\ln\mathcal{L}) = f_{\rm null} - f_{\rm alt}$")
    ax.set_ylabel("Simulation count")

    ax.text(
        0.98,
        0.95,
        (
            fr"$N_{{sim}} = {n_boot}$" "\n"
            fr"$N_{{exceed}} = {n_exceed}$" "\n"
            fr"$p_{{+1}} = {p_plus1:.3g}$"
        ),
        transform=ax.transAxes,
        ha="right",
        va="top",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85),
    )

    ax.legend(loc="best")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(f"/mnt/c/users/Aidan/Documents/plots/028749/{alt_label}.pdf", dpi=300, bbox_inches="tight")
    return fig, ax

In [ ]:
fig, ax = plot_lrt(TS_pexpp, pexpp_summary, alt_label="exp_plaw")
fig, ax = plot_lrt(TS_ppexp, ppexp_summary, alt_label="Pure exponential")
fig, ax = plot_lrt(TS_padd, padd_summary, alt_label="powerlaw+exp")